# Day 8 · Utopia — ประเทศไทยที่ความมืดเป็นของสาธารณะ

**#30DayMapChallenge 2026 · วันที่ 8 พ.ย. 2569** · โจทย์ทางการ: *"A map of something which could be"*

คำถาม: ถ้าอยากให้ประชากรเมือง 90% อยู่ภายใน 30 กม. จากท้องฟ้ามืด ต้องเพิ่มจุดอนุรักษ์ท้องฟ้ามืด
อีกกี่แห่ง และที่ไหน — ใช้ greedy set cover บน hex grid ของ Day 3 (ผู้สมัคร = hex ที่ไกลจากจุดในทะเบียนเกิน 25 กม.)

**ผล: เพิ่ม 15 จุด → จาก 15.2% เป็น 89.7%** · จุดแรกใกล้กรุงเทพจุดเดียวเก็บ 7.5 ล้านคน

> บน Colab สคริปต์จะดึง CSV/geojson จาก GitHub และโหลด shapefile Natural Earth เอง


### ตั้งค่า + shim (Colab/local) + helper ระยะทาง

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 8 · Utopia — "A map of something which could be"
แผนที่ที่อาจเป็นได้: ประเทศไทยที่ *ทุกคน* ข้างในรัศมี 100 กม. จากจุดฟ้ามืด
โดยเพิ่ม "จุดอนุรักษ์ท้องฟ้ามืดแห่งใหม่" ในที่มืดที่ยังไม่มีชื่อในทะเบียน NARIT

ต่อจาก Day 7 (ซึ่งพบว่าแค่ 0.03% ของประเทศเดินถึงจุดฟ้ามืดใน 10 นาที) — วันนี้ถามกลับว่า
"ถ้าอยากให้ความมืดเป็นของสาธารณะ ต้องเพิ่มกี่จุด และเพิ่มตรงไหน"
"""
import os, json, math, time
import numpy as np, pandas as pd
import geopandas as gpd
from shapely.geometry import shape, Point
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection, LineCollection

T0 = time.time()
COLAB = not os.path.isdir(r"D:\GIS30Days\geo-notes")
WORK = os.getcwd() if COLAB else r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "day08_cache") if COLAB else os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day08")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)

import zipfile, urllib.request
def ensure_shp(stem, sub):
    shp = os.path.join(CACHE, stem + ".shp")
    if not os.path.exists(shp):
        zp = os.path.join(CACHE, stem + ".zip")
        urllib.request.urlretrieve(f"https://naciscdn.org/naturalearth/10m/{sub}/{stem}.zip", zp)
        zipfile.ZipFile(zp).extractall(CACHE)
    return shp

def ensure_repo_file(rel):
    dst = os.path.join(CACHE, os.path.basename(rel))
    if COLAB and not os.path.exists(dst):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/" + rel, dst)
    return dst if COLAB else os.path.join(REPO, rel)
for f in sorted(os.listdir(CACHE)):
    if f.endswith(".ttf"): font_manager.fontManager.addfont(os.path.join(CACHE, f))
plt.rcParams["font.family"] = "Trirong"
log = lambda m: print(f"[{time.time()-T0:5.1f}s] {m}", flush=True)

INK, BG, GOLD, CYAN, RED, GREY = "#e8e3d3", "#0b1220", "#f5b942", "#38bdf8", "#f4566a", "#5b6472"
EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")

def hav_km(a, b):
    """a,b: (N,2) arrays of (lat,lon) -> (N,M) km, haversine"""
    la1, lo1 = np.radians(a[:, 1][:, None]), np.radians(a[:, 0][:, None])
    la2, lo2 = np.radians(b[:, 1][None, :]), np.radians(b[:, 0][None, :])
    d = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 2*6371.0*np.arcsin(np.sqrt(d))

### 1. ข้อมูล

In [ ]:
df = pd.read_csv(ensure_repo_file("data/darksky_thailand.csv"))
sites = np.column_stack([df.longitude.values, df.latitude.values])
g3 = json.load(open(ensure_repo_file("posts/day03-polygons/day03_hex_polygons.geojson"), encoding="utf-8"))
hexes = [shape(f["geometry"]) for f in g3["features"]]
kms = np.array([f["properties"]["km_nearest"] for f in g3["features"]])
cen = np.array([[h.centroid.x, h.centroid.y] for h in hexes])
# ผู้สมัคร "จุดใหม่" = ศูนย์กลาง hex ที่ *ไกล* จากจุดในทะเบียนเกิน 25 กม. (= ที่มืดที่ยังไม่มีใครดูแล)
cand_mask = kms > 25.0
cand = cen[cand_mask]
log(f"จุด NARIT {len(sites)} · hex {len(hexes)} · hex ไกลเกิน 25 กม. = {cand_mask.sum()} (ผู้สมัครจุดใหม่)")

places = gpd.read_file(ensure_shp("ne_10m_populated_places", "cultural"))
th = places[(places.ADM0NAME == "Thailand") & (places.POP_MAX > 0)].copy()
th = th[th.LATITUDE.notna() & th.LONGITUDE.notna()]
city = np.column_stack([th.LONGITUDE.values, th.LATITUDE.values])
pop = th.POP_MAX.values.astype(float)
log(f"เมือง {len(th)} แห่ง · ประชากรรวมในชุดข้อมูล = {pop.sum():,.0f} คน (Natural Earth = เฉพาะเมือง)")

countries = gpd.read_file(ensure_shp("ne_10m_admin_0_countries", "cultural"))
TH = gpd.GeoSeries([countries[countries.ISO_A3 == "THA"].geometry.iloc[0]], crs="EPSG:4326").to_crs(EQ)[0]

### 2. สถานะปัจจุบัน + greedy set cover

In [ ]:
RS = [30.0, 60.0, 100.0]                    # กม. — "ครึ่งชั่วโมง / หนึ่งชั่วโมง / คืนนั้น"
R = RS[0]                                  # ค่าหลักของแผนที่
d_city_site = hav_km(city, sites)
d_cand_site = hav_km(cand, sites)          # ผู้สมัคร <-> จุดเดิม (ใช้ตรวจว่าไม่ไปตั้งทับของเดิม)
base = {r: (hav_km(city, sites).min(axis=1) <= r) for r in RS}
for r in RS:
    log(f"ทุกวันนี้มี {base[r].sum()}/{len(th)} เมือง (ประชากร {pop[base[r]].sum()/pop.sum()*100:4.1f}%) "
        f"ที่อยู่ในรัศมี {r:5.0f} กม. จากจุดฟ้ามืดในทะเบียน")

def solve(R, budget=15):
    covered = base[R].copy(); chosen, ladder = [], []
    d_city_cand = hav_km(city, cand)
    for step in range(budget):
        gain = np.array([pop[(~covered) & (d_city_cand[:, j] <= R)].sum() for j in range(len(cand))])
        j = int(np.argmax(gain))
        if gain[j] <= 0: break
        newly = (~covered) & (d_city_cand[:, j] <= R)
        top = int(np.argmax(np.where(newly, pop, 0.0)))
        covered |= (d_city_cand[:, j] <= R)
        chosen.append(dict(idx=j, lat=float(cand[j, 1]), lon=float(cand[j, 0]), gained_pop=int(gain[j]),
                           km_nearest_existing=float(d_cand_site[j].min()),
                           city_around=str(th.NAME.iloc[top]), province=str(th.ADM1NAME.iloc[top])))
        ladder.append((step + 1, int(covered.sum()), float(pop[covered].sum()/pop.sum()*100)))
        log(f"  R={R:.0f} + จุด #{step+1} ใกล้ {chosen[-1]['city_around']:22} "
            f"({cand[j,1]:.3f},{cand[j,0]:.3f}) เก็บเพิ่ม {gain[j]/1e6:5.2f} ล้าน → รวม {pop[covered].sum()/pop.sum()*100:.1f}%")
    return chosen, ladder, covered

chosen, ladder, covered = solve(R)
new = np.array([[c["lon"], c["lat"]] for c in chosen]) if chosen else np.zeros((0, 2))
res60 = solve(60.0, budget=15); res100 = solve(100.0, budget=15)
log(f"สรุป R={R:.0f} กม.: เพิ่ม {len(chosen)} จุด → {pop[covered].sum()/pop.sum()*100:.1f}% "
    f"· R=60: {len(res60[0])} จุด → {res60[2].sum() and pop[res60[2]].sum()/pop.sum()*100:.1f}%"
    f" · R=100: {len(res100[0])} จุด → {pop[res100[2]].sum()/pop.sum()*100:.1f}%")

### 3. verification (กติกาของซีรีส์)

In [ ]:
assert len(chosen) == len({(round(c["lat"],4), round(c["lon"],4)) for c in chosen}), "จุดใหม่ซ้ำกัน"
assert all(c["km_nearest_existing"] > 25 for c in chosen)
assert covered.sum() >= base[R].sum(), "ครอบคลุมลดลง?"

log("ตรวจแล้ว: จุดใหม่ไม่ซ้ำ · ทุกจุดอยู่ไกลจากของเดิม >25 กม. · จำนวนเมืองครอบคลุมไม่ลดลง")

### 4. แผนที่หลัก

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 12), facecolor=BG); ax.set_facecolor(BG)
ax.set_aspect("equal"); ax.axis("off")
def rings(g):
    for poly in (g.geoms if g.geom_type.startswith("Multi") else [g]):
        yield np.asarray(poly.exterior.coords)
ax.add_collection(PolyCollection(list(rings(TH)), facecolor="#111a2b", edgecolor=GREY, lw=0.9))
for c in chosen:
    p = gpd.GeoSeries([Point(c["lon"], c["lat"])], crs="EPSG:4326").to_crs(EQ)[0]
    ring = gpd.GeoSeries([p.buffer(R*1000)], crs=EQ).to_crs("EPSG:4326")[0]
    rr = np.asarray(ring.exterior.coords)
    ax.add_collection(PolyCollection([rr], facecolor=GOLD,
                                     edgecolor=GOLD, lw=0.7, alpha=0.05, linestyle=(0, (3, 2))))
    ax.plot(rr[:, 0], rr[:, 1], color=GOLD, lw=0.55, alpha=0.5, zorder=4)
    ax.plot(p.x, p.y, marker="*", ms=17, color=GOLD, mec="#0b1220", mew=0.8, zorder=7)
for s in sites:
    p = gpd.GeoSeries([Point(s[0], s[1])], crs="EPSG:4326").to_crs(EQ)[0]
    ax.plot(p.x, p.y, marker="o", ms=3.0, color=CYAN, alpha=0.85, zorder=6)
city_xy = gpd.GeoSeries([Point(*c) for c in city], crs="EPSG:4326").to_crs(EQ)
for (x, y), nm, n, ok in zip(zip(city_xy.x.values, city_xy.y.values), th.NAME.values, pop, covered):
    ax.plot(x, y, marker="o", ms=float(np.clip(2.2 + np.sqrt(n)/170, 2.2, 13)),
            color=INK if ok else RED, mec="#0b1220", mew=0.4, alpha=0.95, zorder=5)
    if not ok and n >= 100000:
        ax.annotate(str(nm)[:18], (x, y), textcoords="offset points", xytext=(5, 4),
                    fontsize=6.4, color=RED, alpha=0.92, zorder=5.5)
bx, by, bx1, by1 = TH.bounds
ax.set_xlim(bx-70_000, bx1+70_000); ax.set_ylim(by-70_000, by1+70_000)
ax.set_title(f"Day 8 · Utopia — ประเทศไทยที่ความมืดเป็นของสาธารณะ\n"
             f"เพิ่มจุดฟ้ามืดใหม่ {len(chosen)} แห่ง ให้คนเมือง {pop[covered].sum()/pop.sum()*100:.0f}% "
             f"อยู่ภายใน {R:.0f} กม. จากฟ้ามืด", color=INK, fontsize=13.5, pad=16, fontweight="bold")
# legend เป็น marker จริงใน 4 ช่องคงที่ (Trirong ไม่มี glyph ★/◆)
LEG = [(0.00, "o", RED, 5.0,  f"เมืองที่เข้าไม่ถึง ({int((~covered).sum())}/{len(th)})"),
       (0.26, "o", INK, 6.5,  "เมืองที่เข้าถึงแล้ว (ขนาด = ประชากร)"),
       (0.56, "o", CYAN, 3.4, f"จุดในทะเบียน NARIT ({len(sites)})"),
       (0.80, "*", GOLD, 13.0, f"จุดใหม่ ({len(chosen)}) รัศมี {R:.0f} กม.")]
for xf, mk, cc, ms_, lab in LEG:
    ax.plot([xf], [-0.03], marker=mk, color=cc, ms=ms_, mec="#0b1220", mew=0.5,
            clip_on=False, transform=ax.transAxes, zorder=9)
    ax.text(xf + 0.022, -0.03, lab, transform=ax.transAxes, fontsize=7.6,
            color="#cbd5e1", va="center")

fig.savefig(os.path.join(OUT, "day08_utopia_preview.png"), dpi=150, facecolor=BG, bbox_inches="tight")
log("เขียน day08_utopia_preview.png")

### 5. บันไดความครอบคลุม

In [ ]:
fig2, ax2 = plt.subplots(figsize=(8.6, 4.6), facecolor=BG); ax2.set_facecolor("#0e1626")
xs = [0] + [a for a, b, c in ladder]; ys = [pop[base[R]].sum()/pop.sum()*100] + [c for a, b, c in ladder]
ax2.plot(xs, ys, marker="o", color=GOLD, lw=2.0, ms=6)
ax2.axhline(90, color=RED, ls="--", lw=1.0)
ax2.text(0.15, 90.6, "เป้าหมาย 90%", color=RED, fontsize=8.5)
for a, b, c in ladder[::max(1, len(ladder)//6)]:
    ax2.annotate(f"{c:.0f}%", (a, c), textcoords="offset points", xytext=(0, 8), color="#cbd5e1", fontsize=8, ha="center")
ax2.set_xlabel("จำนวนจุดฟ้ามืดแห่งใหม่ที่เพิ่ม", color="#cbd5e1"); ax2.set_ylabel("สัดส่วนประชากรเมืองที่เข้าถึงได้ (%)", color="#cbd5e1")
ax2.set_title(f"รัศมี {R:.0f} กม. — เพิ่ม {len(chosen)} จุดได้ {ys[-1]:.1f}% (อิ่มตัวแล้ว)", color=INK, fontsize=11.5)
for sp in ax2.spines.values(): sp.set_color(GREY)
ax2.tick_params(colors="#94a3b8"); ax2.grid(alpha=0.15, color=GREY)
fig2.tight_layout(); fig2.savefig(os.path.join(OUT, "day08_ladder.png"), dpi=150, facecolor=BG)
log("เขียน day08_ladder.png")

### 6. ไฟล์ข้อมูล (ต้องถูกอ้างในโพสต์ ไม่งั้น 404)

In [ ]:
gj = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "properties": {"no": i+1, "lat": round(c["lat"], 5), "lon": round(c["lon"], 5),
                                       "gained_pop": c["gained_pop"], "km_to_nearest_existing_site": round(c["km_nearest_existing"], 1)},
     "geometry": {"type": "Point", "coordinates": [c["lon"], c["lat"]]}} for i, c in enumerate(chosen)]}
json.dump(gj, open(os.path.join(OUT, "day08_utopia_sites.geojson"), "w"), ensure_ascii=False, indent=1)
json.dump({"R_km": R, "cities": int(len(th)), "pop_total": int(pop.sum()),
           "covered_before_pct": round(pop[base[R]].sum()/pop.sum()*100, 2),
           "covered_after_pct": round(pop[covered].sum()/pop.sum()*100, 2),
           "new_sites": len(chosen), "ladder": ladder},
          open(os.path.join(OUT, "day08_stats.json"), "w"), ensure_ascii=False, indent=1)
log(f"เขียน geojson + stats · รวมเวลา {time.time()-T0:.1f} วิ")